In [ ]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')

# Create the cursor
cursor = conn.cursor()

# I'm updating activities. I'll add 10 activities
new_activities = ("Budget backpacking", "Beach Vacation", "Cruise Vacation", "Road Tripping", "Camping Trip", "Ski Holiday", "Sightseeing Tour", "Resort Stay", "Food Touring", "Heritage Travel")

# Now we execute the change using the cursor
cursor.execute("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)

new_spotlight_description = ("nothing_x", "nothing_x2", "nothing_x3", "nothing_x4", "nothing_x5", "nothing_x6", "nothing_x7", "nothing_x8", "nothing_x9", "nothing_x10")

cursor.execute("INSERT INTO Destinations_Activities (Spotlight_Description) VALUES (?)", new_spotlight_description)

new_travel_vibe = ("Frugal Adventure", "Sun & Sand", "Nautical Luxury", "Open Road", "Rustic Wilderness", "Alpine Thrills", "Urban Explorer", "Pure Relaxation", "Gastronomic Journey", "Cultural Roots")
cursor.execute("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibe)

# Let's add cities:

cities_to_add = [
("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
("Bucharest", "Romania", "The capital of Romania, it's alright."),
("London", "England", "The capital of England, it's a nice place for pubcrawling."),
("Paris", "France", "The capital of France, iconic for food, art, and cafes."),
("Rome", "Italy", "The capital of Italy, packed with ancient history and incredible pasta."),
("New York", "USA", "The Big Apple, famous for its non-stop energy and Broadway shows."),
("Barcelona", "Spain", "A vibrant coastal city known for stunning architecture and tapas."),
("Bangkok", "Thailand", "A bustling tropical hub famous for street food and ornate temples."),
("Sydney", "Australia", "A gorgeous harbor city with amazing beaches and a laid-back vibe."),
("Cairo", "Egypt", "A historic desert metropolis home to the ancient pyramids.")
]

# Now bcs we have multiple arrays, we need to use a loop to add everything
cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)

conn.commit()
print("Data succesfully saved!")

# Now we close it
conn.close()

ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 10 supplied.

In [2]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')
cursor = conn.cursor()

# 1. ADDING ACTIVITIES
# Formatted as a LIST of tuples so we can use executemany
new_activities = [
    ("Budget backpacking",), ("Beach Vacation",), ("Cruise Vacation",), 
    ("Road Tripping",), ("Camping Trip",), ("Ski Holiday",), 
    ("Sightseeing Tour",), ("Resort Stay",), ("Food Touring",), 
    ("Heritage Travel",)
]

cursor.executemany("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)


# 2. ADDING TRAVEL VIBES
# Formatted as a LIST of tuples
new_travel_vibes = [
    ("Frugal Adventure",), ("Sun & Sand",), ("Nautical Luxury",), 
    ("Open Road",), ("Rustic Wilderness",), ("Alpine Thrills",), 
    ("Urban Explorer",), ("Pure Relaxation",), ("Gastronomic Journey",), 
    ("Cultural Roots",)
]

cursor.executemany("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibes)


# 3. ADDING CITIES
# Your formatting here was already absolutely perfect!
cities_to_add = [
    ("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
    ("Bucharest", "Romania", "The capital of Romania, it's alright."),
    ("London", "England", "The capital of England, it's a nice place for pubcrawling."),
    ("Paris", "France", "The capital of France, iconic for food, art, and cafes."),
    ("Rome", "Italy", "The capital of Italy, packed with ancient history and incredible pasta."),
    ("New York", "USA", "The Big Apple, famous for its non-stop energy and Broadway shows."),
    ("Barcelona", "Spain", "A vibrant coastal city known for stunning architecture and tapas."),
    ("Bangkok", "Thailand", "A bustling tropical hub famous for street food and ornate temples."),
    ("Sydney", "Australia", "A gorgeous harbor city with amazing beaches and a laid-back vibe."),
    ("Cairo", "Egypt", "A historic desert metropolis home to the ancient pyramids.")
]

cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)


# Save and close!
conn.commit()
print("Data successfully saved!")
conn.close()

Data successfully saved!


In [3]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')
cursor = conn.cursor()

# We are linking DestinationID to VibeID
# Example: (1, 7) means Tokyo (1) gets the "Urban Explorer" (7) vibe
# Example: (1, 9) means Tokyo (1) also gets "Gastronomic Journey" (9)

vibe_links = [
    (1, 7), (1, 9), # Tokyo
    (2, 1), (2, 7), # Bucharest 
    (3, 7), (3, 10),# London
    (4, 9), (4, 10),# Paris
    (5, 9), (5, 10),# Rome
    (6, 7),         # New York
    (7, 2), (7, 9), # Barcelona
    (8, 1), (8, 9), # Bangkok
    (9, 2), (9, 8), # Sydney
    (10, 1), (10, 10) # Cairo
]

# Insert the pairs into the purely relational link table
cursor.executemany("""
    INSERT INTO Destination_Vibes (DestinationID, VibeID) 
    VALUES (?, ?)
""", vibe_links)

# Let's also populate Destinations_Activities while we are here!
# This requires 3 pieces of data: DestinationID, ActivityID, and Spotlight_Description
activity_links = [
    (1, 9, "Eat your way through the Tsukiji Outer Market."), # Tokyo -> Food Touring
    (4, 10, "Spend days getting lost in the Louvre and Musée d'Orsay."), # Paris -> Heritage Travel
    (8, 1, "Backpack through Khao San Road for the ultimate budget trip.") # Bangkok -> Budget Backpacking
]

cursor.executemany("""
    INSERT INTO Destinations_Activities (DestinationID, ActivityID, Spotlight_Description) 
    VALUES (?, ?, ?)
""", activity_links)

# Save and close!
conn.commit()
print("Link tables successfully populated!")
conn.close()

Link tables successfully populated!


In [10]:
# Let's visualise it using panda
import sqlite3
import pandas as pd

conn = sqlite3.connect('travel_planner.db')

# The Query: Let's find all the vibes associated with Tokyo
# 1. We SELECT the columns we actually want to read
# 2. We FROM the main table
# 3. We JOIN the tables together by matching the Primary Keys to the Foreign Keys
# 4. We use WHERE to filter down to just City ID 1 (Tokyo)

query = """
    SELECT 
        Destinations.CityName, 
        Travel_Vibes.VibeName
    FROM Destinations
    JOIN Destination_Vibes ON Destinations.DestinationID = Destination_Vibes.DestinationID
    JOIN Travel_Vibes ON Destination_Vibes.VibeID = Travel_Vibes.VibeID
    WHERE Destinations.DestinationID = 4;
"""

# Ask pandas to run it and display it!
df_results = pd.read_sql_query(query, conn)
conn.close()

display(df_results)



,CityName,VibeName
0,Paris,Gastronomic Journey
1,Paris,Cultural Roots
